# SEO Analytics Portfolio — Data Cleaning, Views and Queries using SQL
**Daniel Muyeba** · [LinkedIn](https://www.linkedin.com/in/daniel-muyeba-60093285/) · [GitHub](https://www.github.com/dmuyeba/)

---

## Overview
This notebook is the first step in an SEO data portfolio. The dataset captures an e-commerce site's organic search performance and combines Google Search Console metrics (clicks, impressions, ranking position) with on-page technical metrics (response time, word count, internal links) and page-type taxonomy (product vs categorical vs brands), providing a rich basis for analysis.

### Sections in this Notebook:
#### **1. Data Cleaning and Schema Design**
#### **2. Views**
#####   2.1. **Raw Staging Table** - *stg_page_performance* 
#####   2.2. **Base View**: cleaned types and derived metrics - *vw_page_base*
#####   2.3. **Quick Win Opportunities**: rankning opportunity filter - *vw_quick_wins*
#####   2.4. **Segment Summary**: segment-level KPI rollup - *vw_segments_summary*
#####   2.5. **Position Distribution**: SERP position bucketing - *vw_position_distribution*
#####   2.6. **Crawl Efficiency**: crawl budget waste signals - *vw_crawl_efficiency*
#####   2.7. **Crawl Efficiency**: crawl budget waste signals - *vw_crawl_efficiency*
#####   2.8. **Content Health Audit**: on-page quality flags - *vw_content_health*

#### **3. Queries**
#####   3.1. - **Site-Level Overview**
#####   3.2. - **Quick Win Opportunities**
#####   3.3. - **Internal Linking & Link Equity**
#####   3.4. - **Crawl Budget Analysis**
#####   3.5. - **Content & Technical Health**
#####   3.6. - **Engagement Exploration**
#####   3.7. - **Data Export Queries**

#### **4. Recommended Actions**
#####    4.1 - **Immediate**
#####    4.2 - **Short-Term**
#####    4.3 - **Medium-Term**
#####    4.4 - **Long-Term**
---

*For predictive modelling including click prediction and K-Means clustering of opportunity segments using Python, see the notebook and report titled **SEO_Portfolio_Py** and **SEO_Portfolio_Py_Report**. For inferential statistical analysis and hypothesis testing using R, see **SEO_Portfolio_R** and **SEO_Portfolio_R_Report**.*

---

## 1. Data Cleaning, Views and Queries - SQL

### 1.1. Schema
A raw staging table that mirrors the source data export exactly to clean and transform the data in views using DROP and CREATE to create tables and views looking at user engagement signals, page taxonomy, on-page content signals, and technical SEO signals for use in analysis later on.

In [300]:
!pip install ipython-sql
!pip install sqlalchemy

import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
import os

%config SqlMagic.autopandas = True

In [302]:
engine = create_engine("sqlite:///seo_data.db")

In [304]:
df = pd.read_csv("data.csv", sep=";")

In [306]:
df.to_sql("mytable", con=engine, if_exists="replace", index=False)
os.makedirs("sql_report_tables", exist_ok = True)

table_counter = 1

def save_table(table):
    global table_counter

    filename = f"sql_report_tables/table_{table_counter}.html"

    # SQL magic
    if hasattr(table, "DataFrame"):
        df = table.DataFrame()

    # Pandas DataFrame
    else:
        df = table

    df.to_html(filename, index=False)

    print(f"Saved: {filename}")

    table_counter += 1

In [308]:
%load_ext sql
%sql sqlite:///seo_data.db
df.head()

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


,Clicks,Impressions,Position,BounceRate,ViewDepth,TimeSpent,RobotsVisits,Mobility,Segments,Title Length,Meta Description Length,H1 Length,Word Count,Sentence Count,Folder Depth,Link Score,Inlinks,Outlinks,Response Time
0,1939,41950,"8,6","0,099963045","2,657335551",00:11:30,837.0,"0,271618625",NaN,83.0,188.0,0.0,1553.0,367.0,0.0,99.0,24095.0,334.0,"2,185"
1,1798,27042,"8,2","0,314511575","1,15471485",00:00:41,7.0,"0,869000565",product,152.0,227.0,37.0,950.0,377.0,3.0,2.0,173.0,77.0,"1,332"
2,951,18076,"6,3","0,33880422","1,25908558",00:00:53,3.0,"0,852286049",product,101.0,159.0,38.0,858.0,341.0,3.0,1.0,143.0,77.0,"1,759"
3,863,10697,"7,9","0,262893983","1,348853868",00:00:43,3.0,"0,829512894",product,70.0,137.0,24.0,910.0,359.0,3.0,1.0,143.0,77.0,"1,495"
4,822,7269,"17,8","0,221176471","3,207843137",00:02:19,27.0,"0,534117647",NaN,85.0,181.0,24.0,528.0,103.0,2.0,2.0,120.0,334.0,"0,18"


In [310]:
csv_path = Path("data.csv")
df_csv = pd.read_csv(csv_path, sep=";")

DB_PATH = Path("..")/"seo_data.db"
conn = sqlite3.connect(DB_PATH)
print('Connected to:', DB_PATH.resolve())

Connected to: C:\Users\dmuye\seo_data.db


In [316]:
numeric_cols = [
    "BounceRate", "ViewDepth", "RobotsVisits", "Mobility"
]

for col in numeric_cols:
    df_csv[col] = pd.to_numeric(df_csv[col], errors="coerce")

df_csv = df_csv.reset_index(drop=True)
df_csv["page_id"] = df_csv.index + 1

In [320]:
df_csv.to_csv("df_csv.csv", index=False)
df_csv.head()

,Clicks,Impressions,Position,BounceRate,ViewDepth,TimeSpent,RobotsVisits,Mobility,Segments,Title Length,Meta Description Length,H1 Length,Word Count,Sentence Count,Folder Depth,Link Score,Inlinks,Outlinks,Response Time,page_id
0,1939,41950,"8,6",NaN,NaN,00:11:30,837.0,NaN,NaN,83.0,188.0,0.0,1553.0,367.0,0.0,99.0,24095.0,334.0,"2,185",1
1,1798,27042,"8,2",NaN,NaN,00:00:41,7.0,NaN,product,152.0,227.0,37.0,950.0,377.0,3.0,2.0,173.0,77.0,"1,332",2
2,951,18076,"6,3",NaN,NaN,00:00:53,3.0,NaN,product,101.0,159.0,38.0,858.0,341.0,3.0,1.0,143.0,77.0,"1,759",3
3,863,10697,"7,9",NaN,NaN,00:00:43,3.0,NaN,product,70.0,137.0,24.0,910.0,359.0,3.0,1.0,143.0,77.0,"1,495",4
4,822,7269,"17,8",NaN,NaN,00:02:19,27.0,NaN,NaN,85.0,181.0,24.0,528.0,103.0,2.0,2.0,120.0,334.0,"0,18",5


In [321]:
%%sql 
sqlite:///seo_data.db

DROP TABLE IF EXISTS stg_page_performance

Done.


""


In [322]:
%%sql
CREATE TABLE stg_page_performance (
    -- Surrogate primary key
    page_id INTEGER PRIMARY KEY,

    -- SEARCH PERFORMANCE SIGNALS
    clicks INTEGER NOT NULL DEFAULT 0,
    impressions INTEGER NOT NULL DEFAULT 0,
    position REAL CHECK (position > 0),

    -- USER ENGAGEMENT SIGNALS
    bounce_rate REAL CHECK (bounce_rate BETWEEN 0 AND 1),
    view_depth REAL CHECK (view_depth >= 0),
    time_spent TEXT,
    robots_visits INTEGER DEFAULT 0,
    mobility REAL CHECK (mobility BETWEEN 0 AND 1),

    -- PAGE TAXONOMY
    segments TEXT CHECK (segments IN ('product', 'catalog', 'brands', 'uncategorised')),

    -- ON-PAGE CONTENT SIGNALS
    title_length INTEGER CHECK (title_length >= 0),
    meta_description_length INTEGER CHECK (meta_description_length >= 0),
    h1_length INTEGER CHECK (h1_length >= 0),
    word_count INTEGER CHECK (word_count >= 0),
    sentence_count INTEGER CHECK (sentence_count >= 0),

    -- TECHNICAL SEO SIGNALS
    folder_depth INTEGER CHECK (folder_depth >= 0),
    link_score REAL CHECK (link_score BETWEEN 0 AND 99),
    inlinks INTEGER CHECK (inlinks >= 0),
    outlinks INTEGER CHECK (outlinks >= 0),
    response_time REAL CHECK (response_time >= 0)
);

 * sqlite:///seo_data.db
Done.


""


In [347]:
%%sql ALTER TABLE stg_page_performance
ADD COLUMN position_bucket TEXT;

 * sqlite:///seo_data.db
Done.


""


In [348]:
%%sql
UPDATE stg_page_performance
SET position_bucket = 
    CASE
        WHEN position BETWEEN 1 AND 3 THEN '1–3'
        WHEN position BETWEEN 4 AND 10 THEN '4–10'
        WHEN position BETWEEN 11 AND 20 THEN '11–20'
        WHEN position BETWEEN 21 AND 50 THEN '21–50'
        WHEN position > 50 THEN '50+'
        ELSE 'Unknown'
    END;

 * sqlite:///seo_data.db
0 rows affected.


""


In [349]:
conn = sqlite3.connect("seo_data.db")

schema = pd.read_sql("PRAGMA table_info(stg_page_performance);", conn)
schema.to_csv("schema.csv", index=False)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,page_id,INTEGER,0,None,1
1,1,clicks,INTEGER,1,0,0
2,2,impressions,INTEGER,1,0,0
3,3,position,REAL,0,None,0
4,4,bounce_rate,REAL,0,None,0
5,5,view_depth,REAL,0,None,0
6,6,time_spent,TEXT,0,None,0
7,7,robots_visits,INTEGER,0,0,0
8,8,mobility,REAL,0,None,0
9,9,segments,TEXT,0,None,0


In [350]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("seo_data.db")

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)


,name
0,mytable
1,stg_page_performance


In [351]:
df_csv = df_csv.rename(columns={
    "Clicks": "clicks",
    "Impressions": "impressions",
    "Position": "position",
    "BounceRate": "bounce_rate",
    "ViewDepth": "view_depth",
    "TimeSpent": "time_spent",
    "RobotsVisits": "robots_visits",
    "Mobility": "mobility",
    "Segments": "segments",
    "Title Length": "title_length",
    "Meta Description Length": "meta_description_length",
    "H1 Length": "h1_length",
    "Word Count": "word_count",
    "Sentence Count": "sentence_count",
    "Folder Depth": "folder_depth",
    "Link Score": "link_score",
    "Inlinks": "inlinks",
    "Outlinks": "outlinks",
    "Response Time": "response_time"
})

In [352]:
df_csv[df_csv["bounce_rate"].isna()]

,clicks,impressions,position,bounce_rate,view_depth,time_spent,robots_visits,mobility,segments,title_length,meta_description_length,h1_length,word_count,sentence_count,folder_depth,link_score,inlinks,outlinks,response_time,page_id
0,1939,41950,"8,6",NaN,NaN,00:11:30,837.0,NaN,NaN,83.0,188.0,0.0,1553.0,367.0,0.0,99.0,24095.0,334.0,"2,185",1
1,1798,27042,"8,2",NaN,NaN,00:00:41,7.0,NaN,product,152.0,227.0,37.0,950.0,377.0,3.0,2.0,173.0,77.0,"1,332",2
2,951,18076,"6,3",NaN,NaN,00:00:53,3.0,NaN,product,101.0,159.0,38.0,858.0,341.0,3.0,1.0,143.0,77.0,"1,759",3
3,863,10697,"7,9",NaN,NaN,00:00:43,3.0,NaN,product,70.0,137.0,24.0,910.0,359.0,3.0,1.0,143.0,77.0,"1,495",4
4,822,7269,"17,8",NaN,NaN,00:02:19,27.0,NaN,NaN,85.0,181.0,24.0,528.0,103.0,2.0,2.0,120.0,334.0,"0,18",5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9434,0,12,"7,2",NaN,NaN,NaN,NaN,NaN,brands,103.0,221.0,18.0,725.0,264.0,3.0,1.0,184.0,158.0,"0,797",9435
9435,0,2,3,NaN,NaN,NaN,NaN,NaN,brands,115.0,227.0,24.0,555.0,201.0,3.0,2.0,235.0,146.0,"0,487",9436
9436,0,3,56,NaN,NaN,NaN,NaN,NaN,brands,103.0,221.0,18.0,419.0,174.0,3.0,1.0,24.0,139.0,"0,475",9437
9437,0,1,5,NaN,NaN,NaN,NaN,NaN,brands,105.0,222.0,19.0,653.0,212.0,3.0,1.0,75.0,143.0,"0,51",9438


In [353]:
df_csv["bounce_rate"].unique()[:20]
df_csv[df_csv["bounce_rate"].isna()].shape
df_csv[df_csv["bounce_rate"].apply(lambda x: isinstance(x, str))].head()


,clicks,impressions,position,bounce_rate,view_depth,time_spent,robots_visits,mobility,segments,title_length,meta_description_length,h1_length,word_count,sentence_count,folder_depth,link_score,inlinks,outlinks,response_time,page_id


In [354]:
df_csv.to_sql("stg_page_performance", conn, if_exists="append", index=False)

9439

In [355]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [356]:
%%sql
PRAGMA table_info(stg_page_performance);

 * sqlite:///seo_data.db
Done.


,cid,name,type,notnull,dflt_value,pk
0,0,page_id,INTEGER,0,None,1
1,1,clicks,INTEGER,1,0,0
2,2,impressions,INTEGER,1,0,0
3,3,position,REAL,0,None,0
4,4,bounce_rate,REAL,0,None,0
5,5,view_depth,REAL,0,None,0
6,6,time_spent,TEXT,0,None,0
7,7,robots_visits,INTEGER,0,0,0
8,8,mobility,REAL,0,None,0
9,9,segments,TEXT,0,None,0


In [357]:
%%sql
-- INDEXES

CREATE INDEX IF NOT EXISTS idx_stg_segments
    ON stg_page_performance (segments);

CREATE INDEX IF NOT EXISTS idx_stg_position
    ON stg_page_performance (position);

CREATE INDEX IF NOT EXISTS idx_stg_clicks_impressions
    ON stg_page_performance (clicks, impressions);

CREATE INDEX IF NOT EXISTS idx_stg_inlinks
    ON stg_page_performance (inlinks);

 * sqlite:///seo_data.db
Done.
Done.
Done.
Done.


""


In [358]:
pd.read_sql("PRAGMA table_info(stg_page_performance);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,page_id,INTEGER,0,None,1
1,1,clicks,INTEGER,1,0,0
2,2,impressions,INTEGER,1,0,0
3,3,position,REAL,0,None,0
4,4,bounce_rate,REAL,0,None,0
5,5,view_depth,REAL,0,None,0
6,6,time_spent,TEXT,0,None,0
7,7,robots_visits,INTEGER,0,0,0
8,8,mobility,REAL,0,None,0
9,9,segments,TEXT,0,None,0


### 1.2. Views:

#### 1.2.1. Base View
Purpose: to clean types, standardise NULLs and compute fundamental derived metrics. All downstream views reference this view, rather than the raw staging table, so any cleaning fixes here propogate everywhere else automatically.

In [361]:
%%sql
DROP VIEW IF EXISTS vw_page_base;

CREATE VIEW vw_page_base AS

SELECT
    page_id,

    -- Search performance
    clicks,
    impressions,
    position,

    -- Calculated CTR (guarded against zero division for rows with 0 impressions
    CASE
        WHEN impressions > 0 THEN ROUND(CAST(clicks AS REAL) / impressions, 4)
        ELSE NULL
    END AS ctr,

    -- Engagement signals (crawl-only rows may be NULL)
    bounce_rate,
    view_depth,
    time_spent,

    -- Converting time_spent to total seconds
    -- Splits on ':' and weight hours x 3600, minutes x 60, seconds x 1
    -- Returns NULL if time_spent is NULL or formatted wrong
    CASE
        WHEN time_spent IS NOT NULL
            AND time_spent GLOB '[0-9][0-9]:[0-9][0-9]:[0-9][0-9]'
        THEN (
            CAST(SUBSTR(time_spent, 1, 2) AS INTEGER) * 3600 +
            CAST(SUBSTR(time_spent, 4, 2) AS INTEGER) * 60  +
            CAST(SUBSTR(time_spent, 7, 2) AS INTEGER)
        )
        ELSE NULL
    END AS time_spent_seconds,

    robots_visits,
    mobility,

    -- Page taxonomy
    COALESCE(segments, 'uncategorised') AS segment,

    --On-page content signals (analytics-only rows may be NULL)
    title_length,
    meta_description_length,
    h1_length,
    word_count,
    sentence_count,

    -- Technical signals
    folder_depth,
    link_score,
    inlinks,
    outlinks,
    response_time,

    -- SERP position buckets:
    -- Top 3 = featured/branded
    -- 4-10 = other page 1
    -- 11-20 = page 2/quick-win territory
    -- 21-50 = middle ground/needs work
    -- 51+ = essentially invisible
    CASE
        WHEN position BETWEEN 1  AND 3   THEN '1. Top 3'
        WHEN position BETWEEN 4  AND 10  THEN '2. Page 1 (4-10)'
        WHEN position BETWEEN 11 AND 20  THEN '3. Page 2 (11-20)'
        WHEN position BETWEEN 21 AND 50  THEN '4. Mid (21-50)'
        WHEN position > 50               THEN '5. Deep (51+)'
        ELSE                                  'Unknown'
    END AS position_bucket,

    -- Content length quality flags/best-practice thresholds
    -- Title - 50-60 characters; <30 or >70 is flagged
    CASE
        WHEN title_length IS NULL        THEN 'missing'
        WHEN title_length < 30           THEN 'too_short'
        WHEN title_length BETWEEN 30 AND 60 THEN 'optimal'
        WHEN title_length > 60           THEN 'too_long'
    END AS title_length_flag,

    -- Meta description - 120-158 characters
    CASE
        WHEN meta_description_length IS NULL          THEN 'missing'
        WHEN meta_description_length < 70             THEN 'too_short'
        WHEN meta_description_length BETWEEN 70 AND 158 THEN 'optimal'
        WHEN meta_description_length > 158            THEN 'too_long'
    END AS meta_desc_flag,

    -- H1 - present = 1+, absent = 0
    CASE
        WHEN h1_length IS NULL THEN 'missing'
        WHEN h1_length = 0     THEN 'absent'
        ELSE                        'present'
    END AS h1_flag,

    -- Response time classification (seconds)
    -- <0.8s = fast (good TTFB), 0.8-2s = acceptable, >2s = slow
    CASE
        WHEN response_time IS NULL  THEN 'unknown'
        WHEN response_time < 0.8    THEN 'fast'
        WHEN response_time <= 2.0   THEN 'acceptable'
        ELSE                             'slow'
    END AS response_time_flag

FROM stg_page_performance;

 * sqlite:///seo_data.db
Done.
Done.


""


#### 1.2.2 Quick Win Opportunities
Purpose: to show pages in positions 4-20 with sufficient search volumne that would benefit from CTR optimisation such as title and meta description optimising, or minor content improveemnts. These are the pages with the highest ROI.

In [363]:
%%sql
DROP VIEW IF EXISTS vw_quick_wins;

CREATE VIEW vw_quick_wins AS

SELECT
    page_id,
    segment,
    clicks,
    impressions,
    position,
    ctr,
    position_bucket,

    -- Expected CTR benchmarks by position (industry averages, rounded)
    -- Used to surface pages underperforming vs their ranking peers
    CASE
        WHEN position BETWEEN 1  AND 1  THEN 0.28
        WHEN position BETWEEN 2  AND 2  THEN 0.15
        WHEN position BETWEEN 3  AND 3  THEN 0.11
        WHEN position BETWEEN 4  AND 5  THEN 0.08
        WHEN position BETWEEN 6  AND 10 THEN 0.05
        WHEN position BETWEEN 11 AND 20 THEN 0.02
        ELSE                                 0.01
    END AS expected_ctr_benchmark,

    -- Actual vs expected CTR gap (negative = underperforming)
    ROUND(
        ctr - CASE
            WHEN position BETWEEN 1  AND 1  THEN 0.28
            WHEN position BETWEEN 2  AND 2  THEN 0.15
            WHEN position BETWEEN 3  AND 3  THEN 0.11
            WHEN position BETWEEN 4  AND 5  THEN 0.08
            WHEN position BETWEEN 6  AND 10 THEN 0.05
            WHEN position BETWEEN 11 AND 20 THEN 0.02
            ELSE                                 0.01
        END
    , 4) AS ctr_gap,

    -- Estimated click uplift if CTR reached benchmark (headline metric)
    ROUND(
        impressions * (
            CASE
                WHEN position BETWEEN 1  AND 1  THEN 0.28
                WHEN position BETWEEN 2  AND 2  THEN 0.15
                WHEN position BETWEEN 3  AND 3  THEN 0.11
                WHEN position BETWEEN 4  AND 5  THEN 0.08
                WHEN position BETWEEN 6  AND 10 THEN 0.05
                WHEN position BETWEEN 11 AND 20 THEN 0.02
                ELSE                                 0.01
            END - ctr
        )
    , 0) AS estimated_click_uplift,

    -- On-page signals to help prioritise which pages to fix first
    title_length,
    title_length_flag,
    meta_desc_flag,
    h1_flag,
    word_count,
    inlinks,
    link_score

FROM vw_page_base

WHERE
    -- Position range most likely to benefit from CTR optimisation
    position BETWEEN 4 AND 20

    -- Enough search volume to be worth the effort
    AND impressions >= 100

    -- Exclude pages already performing above benchmark (not a quick win)
    AND ctr < CASE
        WHEN position BETWEEN 4  AND 5  THEN 0.08
        WHEN position BETWEEN 6  AND 10 THEN 0.05
        WHEN position BETWEEN 11 AND 20 THEN 0.02
        ELSE 0.01
    END

ORDER BY estimated_click_uplift DESC;

 * sqlite:///seo_data.db
Done.
Done.


""


#### 1.2.3 Segment Summary
Purpose: to aggregate KPIs rolled up by page type - product, branded, categorical (catalog).

In [365]:
%%sql
DROP VIEW IF EXISTS vw_segment_summary;

CREATE VIEW vw_segment_summary AS

SELECT
    segment,

    -- Volume
    COUNT(*)                                        AS total_pages,
    SUM(clicks)                                     AS total_clicks,
    SUM(impressions)                                AS total_impressions,

    -- Performance averages
    ROUND(AVG(position), 2)                         AS avg_position,
    ROUND(AVG(ctr), 4)                              AS avg_ctr,
    ROUND(CAST(SUM(clicks) AS REAL)
          / NULLIF(SUM(impressions), 0), 4)         AS blended_ctr,

    -- Engagement (only rows with analytics data)
    ROUND(AVG(bounce_rate), 3)                      AS avg_bounce_rate,
    ROUND(AVG(view_depth), 2)                       AS avg_view_depth,
    ROUND(AVG(time_spent_seconds), 0)               AS avg_time_spent_seconds,

    -- Technical health
    ROUND(AVG(response_time), 3)                    AS avg_response_time_s,
    ROUND(AVG(inlinks), 1)                          AS avg_inlinks,
    ROUND(AVG(link_score), 2)                       AS avg_link_score,

    -- Zero-click pages (crawl waste or ranking-but-not-winning)
    SUM(CASE WHEN clicks = 0 THEN 1 ELSE 0 END)    AS zero_click_pages,
    ROUND(
        100.0 * SUM(CASE WHEN clicks = 0 THEN 1 ELSE 0 END) / COUNT(*)
    , 1)                                            AS pct_zero_click,

    -- Quick win candidates within this segment
    SUM(CASE
        WHEN position BETWEEN 4 AND 20
         AND impressions >= 100
        THEN 1 ELSE 0
    END)                                            AS quick_win_candidates,

    -- Content health flags
    SUM(CASE WHEN title_length_flag  = 'optimal'   THEN 1 ELSE 0 END) AS titles_optimal,
    SUM(CASE WHEN meta_desc_flag     = 'optimal'   THEN 1 ELSE 0 END) AS meta_descs_optimal,
    SUM(CASE WHEN h1_flag            = 'present'   THEN 1 ELSE 0 END) AS h1_present,
    SUM(CASE WHEN response_time_flag = 'slow'      THEN 1 ELSE 0 END) AS slow_pages

FROM vw_page_base

GROUP BY segment

ORDER BY total_clicks DESC;

 * sqlite:///seo_data.db
Done.
Done.


""


#### 1.2.4 Position Distribution
Purpose: to show SERP ranking distribution across position buckets, split by segment, and show the shape of the site's ranking profile.

In [367]:
%%sql
DROP VIEW IF EXISTS vw_position_distribution;

CREATE VIEW vw_position_distribution AS

SELECT
    segment,
    position_bucket,
    COUNT(*)                                            AS page_count,
    SUM(clicks)                                         AS total_clicks,
    SUM(impressions)                                    AS total_impressions,
    ROUND(AVG(ctr), 4)                                  AS avg_ctr,
    ROUND(AVG(position), 2)                             AS avg_position,

    -- Share of this segment's total clicks in each bucket
    ROUND(
        100.0 * SUM(clicks)
        / NULLIF(SUM(SUM(clicks)) OVER (PARTITION BY segment), 0)
    , 1)                                                AS pct_segment_clicks,

    -- Share of total site clicks (cross-segment context)
    ROUND(
        100.0 * SUM(clicks)
        / NULLIF(SUM(SUM(clicks)) OVER (), 0)
    , 1)                                                AS pct_total_clicks

FROM vw_page_base

GROUP BY segment, position_bucket

ORDER BY segment, position_bucket;

 * sqlite:///seo_data.db
Done.
Done.


""


#### 1.2.5 Crawl Efficieny
Purpose: to identify pages consuming crawl budget which don't generate organic value; high robots visits and zero clicks means high likelihood of crawl waste. This can inform robots.txt or noindex decisions, and highilight orphaned and lacking content.

In [369]:
%%sql
DROP VIEW IF EXISTS vw_crawl_efficiency;

CREATE VIEW vw_crawl_efficiency AS

SELECT
    page_id,
    segment,
    clicks,
    impressions,
    position,
    robots_visits,

    -- Clicks generated per robot visit (efficiency ratio)
    -- NULL if robots_visits = 0 (no crawl data for this page)
    CASE
        WHEN robots_visits > 0
        THEN ROUND(CAST(clicks AS REAL) / robots_visits, 4)
        ELSE NULL
    END AS clicks_per_robot_visit,

    -- Waste classification
    CASE
        WHEN robots_visits = 0                          THEN 'not_crawled'
        WHEN clicks = 0 AND robots_visits > 10          THEN 'high_waste'
        WHEN clicks = 0 AND robots_visits BETWEEN 1 AND 10 THEN 'low_waste'
        WHEN clicks > 0 AND robots_visits > 0
             AND (CAST(clicks AS REAL) / robots_visits) < 0.1
                                                        THEN 'inefficient'
        ELSE                                                 'efficient'
    END AS crawl_efficiency_flag,

    -- Contextual signals to explain waste
    word_count,
    inlinks,
    folder_depth,
    response_time_flag

FROM vw_page_base

ORDER BY robots_visits DESC, clicks ASC;

 * sqlite:///seo_data.db
Done.
Done.


""


#### 1.2.6 Content Health Audit
Purpose: to flag on-page issues across all crawled pages; designed to replicate the output of a manual SEO content audit, prioritised by the pages with the most impressions.

In [371]:
%%sql
DROP VIEW IF EXISTS vw_content_health;

CREATE VIEW vw_content_health AS

SELECT
    page_id,
    segment,
    clicks,
    impressions,
    position,
    position_bucket,

    -- Raw measurements
    title_length,
    meta_description_length,
    h1_length,
    word_count,
    sentence_count,
    response_time,
    inlinks,
    link_score,
    folder_depth,

    -- Quality flags (from vw_page_base)
    title_length_flag,
    meta_desc_flag,
    h1_flag,
    response_time_flag,

    -- Word count classification (thin / adequate / rich content)
    CASE
        WHEN word_count IS NULL      THEN 'unknown'
        WHEN word_count < 200        THEN 'thin'
        WHEN word_count BETWEEN 200 AND 600 THEN 'adequate'
        ELSE                              'rich'
    END AS content_depth_flag,

    -- Internal link equity flag
    CASE
        WHEN inlinks IS NULL         THEN 'unknown'
        WHEN inlinks = 0             THEN 'orphaned'
        WHEN inlinks < 5             THEN 'poorly_linked'
        ELSE                              'well_linked'
    END AS internal_link_flag,

    -- Total count of issues on this page (useful for sorting / triage)
    (
        CASE WHEN title_length_flag  != 'optimal'      THEN 1 ELSE 0 END +
        CASE WHEN meta_desc_flag     != 'optimal'      THEN 1 ELSE 0 END +
        CASE WHEN h1_flag             = 'absent'       THEN 1 ELSE 0 END +
        CASE WHEN response_time_flag  = 'slow'         THEN 1 ELSE 0 END +
        CASE WHEN word_count < 200
              AND word_count IS NOT NULL               THEN 1 ELSE 0 END +
        CASE WHEN inlinks = 0
              AND inlinks IS NOT NULL                  THEN 1 ELSE 0 END
    ) AS issue_count

FROM vw_page_base

-- Only include pages that have been crawled (have on-page data)
WHERE title_length IS NOT NULL

ORDER BY issue_count DESC, impressions DESC;

 * sqlite:///seo_data.db
Done.
Done.


""


### 1.3 Analysis Queries
Self-contained analytical queries for exploration and reporting, referencing the above views and highlighting the business questions answered. This section is split into the following subsections:

##### 1.3.1 - **Site-Level Overview**
##### 1.3.2 - **Quick Win Opportunities**
##### 1.3.3 - **Internal Linking & Link Equity**
##### 1.3.4 - **Crawl Budget Analysis**
##### 1.3.5 - **Content & Technical Health**
##### 1.3.6 - **Engagement Exploration**
##### 1.3.7 - **Data Export Queries**

#### 1.3.1 Site-Level Overview

In [374]:
# Question 1: What does the site's overall search performance look like?

query = """
SELECT
    COUNT(*) AS total_pages,
    SUM(clicks) AS total_clicks,
    SUM(impressions) AS total_impressions,

    ROUND(
        CAST(SUM(clicks) AS REAL)
        / NULLIF(SUM(impressions), 0) * 100,
        2
    ) AS overall_ctr_pct,

    ROUND(AVG(position), 2) AS avg_position,

    SUM(
        CASE
            WHEN clicks = 0 THEN 1
            ELSE 0
        END
    ) AS zero_click_pages,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN clicks = 0 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        1
    ) AS pct_zero_click,

    SUM(
        CASE
            WHEN position BETWEEN 4 AND 20
             AND impressions >= 100
            THEN 1
            ELSE 0
        END
    ) AS quick_win_candidates

FROM vw_page_base;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_1.html


,total_pages,total_clicks,total_impressions,overall_ctr_pct,avg_position,zero_click_pages,pct_zero_click,quick_win_candidates
0,9439,56444,3020134,1.87,14.08,5121,54.3,213


In [375]:
# Question 2: How do product, catalog and brand pages compare on each KPI?

query = """
SELECT * FROM vw_segment_summary;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_2.html


,segment,total_pages,total_clicks,total_impressions,avg_position,avg_ctr,blended_ctr,avg_bounce_rate,avg_view_depth,avg_time_spent_seconds,avg_response_time_s,avg_inlinks,avg_link_score,zero_click_pages,pct_zero_click,quick_win_candidates,titles_optimal,meta_descs_optimal,h1_present,slow_pages
0,product,3504,26220,1132768,10.67,0.0213,0.0231,0.0,1.06,54.0,1.260,141.2,1.09,1648,47.0,104,11,459,3503,3503
1,catalog,4811,19306,1516421,16.74,0.0164,0.0127,0.0,1.45,58.0,1.862,227.6,1.47,2958,61.5,86,9,19,4801,4806
2,uncategorised,414,9228,277095,12.81,0.0500,0.0333,1.0,1.65,83.0,0.203,1566.2,6.58,87,21.0,11,9,96,149,177
3,brands,710,1690,93850,13.61,0.0216,0.0180,0.0,2.00,83.0,0.030,389.0,1.80,428,60.3,12,33,10,673,710


In [376]:
# Question 3: What proporttion of pages rank on page 1, 2, 3 etc.?

query = """
SELECT
    position_bucket,
    COUNT(*)                                    AS pages,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_pages,
    SUM(clicks)                                 AS clicks,
    ROUND(100.0 * SUM(clicks) / NULLIF(SUM(SUM(clicks)) OVER (), 0), 1) AS pct_clicks,
    ROUND(AVG(ctr) * 100, 2)                    AS avg_ctr_pct
FROM vw_page_base
GROUP BY position_bucket
ORDER BY position_bucket;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_3.html


,position_bucket,pages,pct_pages,clicks,pct_clicks,avg_ctr_pct
0,1. Top 3,108,1.1,28,0.0,6.31
1,2. Page 1 (4-10),955,10.1,2639,4.7,2.38
2,3. Page 2 (11-20),369,3.9,1860,3.3,1.32
3,4. Mid (21-50),278,2.9,330,0.6,1.02
4,5. Deep (51+),7729,81.9,51587,91.4,1.97


#### 1.3.2 Quick Win Opportunities

In [378]:
# Question 4: Which pages would drive the most additional clicks if CTR reached benchmark?

query = """
SELECT
    page_id,
    segment,
    impressions,
    ROUND(position, 1)                          AS position,
    ROUND(ctr * 100, 2)                         AS actual_ctr_pct,
    ROUND(expected_ctr_benchmark * 100, 2)      AS benchmark_ctr_pct,
    ROUND(ctr_gap * 100, 2)                     AS ctr_gap_pct,
    estimated_click_uplift,
    title_length_flag,
    meta_desc_flag,
    h1_flag
FROM vw_quick_wins
LIMIT 25;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_4.html


,page_id,segment,impressions,position,actual_ctr_pct,benchmark_ctr_pct,ctr_gap_pct,estimated_click_uplift,title_length_flag,meta_desc_flag,h1_flag
0,24,catalog,11565,8.0,2.14,5.0,-2.86,331.0,too_long,too_long,present
1,14,catalog,37429,18.0,1.23,2.0,-0.77,288.0,too_long,too_long,present
2,598,product,6006,7.0,0.28,5.0,-4.72,283.0,too_long,too_long,present
3,128,catalog,4234,10.0,1.75,5.0,-3.25,138.0,too_long,too_long,present
4,343,product,3241,7.0,0.93,5.0,-4.07,132.0,too_long,too_long,present
5,196,product,3020,8.0,1.75,5.0,-3.25,98.0,too_long,too_long,present
6,54,catalog,4060,10.0,3.35,5.0,-1.65,67.0,too_long,too_long,present
7,183,catalog,6084,12.0,0.92,2.0,-1.08,66.0,too_long,too_long,present
8,2487,product,1317,8.0,0.15,5.0,-4.85,64.0,too_long,too_long,present
9,473,product,4134,20.0,0.53,2.0,-1.47,61.0,too_long,optimal,present


In [379]:
# Question 5: Which type of pages has the most concentrated opportunity, where are the biggest gains?

query = """
SELECT
    segment,
    COUNT(*)                                    AS candidate_pages,
    SUM(estimated_click_uplift)                 AS total_click_uplift,
    ROUND(AVG(estimated_click_uplift), 0)       AS avg_click_uplift_per_page,
    ROUND(AVG(ctr_gap * 100), 2)               AS avg_ctr_gap_pct,
    SUM(CASE WHEN title_length_flag != 'optimal' THEN 1 ELSE 0 END) AS suboptimal_titles,
    SUM(CASE WHEN meta_desc_flag    != 'optimal' THEN 1 ELSE 0 END) AS suboptimal_metas
FROM vw_quick_wins
GROUP BY segment
ORDER BY total_click_uplift DESC;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_5.html


,segment,candidate_pages,total_click_uplift,avg_click_uplift_per_page,avg_ctr_gap_pct,suboptimal_titles,suboptimal_metas
0,catalog,72,1529.0,21.0,-2.22,72,72
1,product,88,1526.0,17.0,-2.60,88,68
2,uncategorised,6,81.0,14.0,-2.10,6,4
3,brands,8,53.0,7.0,-2.25,8,8


In [380]:
# Question 6: Which pages have high impressions, appear in positions 11-20 and could be nudged up?

query = """
SELECT
    page_id,
    segment,
    impressions,
    ROUND(position, 1)                          AS position,
    ROUND(ctr * 100, 2)                         AS ctr_pct,
    estimated_click_uplift,
    inlinks,
    link_score,
    word_count
FROM vw_quick_wins
WHERE position BETWEEN 11 AND 20
ORDER BY impressions DESC
LIMIT 20;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_6.html


,page_id,segment,impressions,position,ctr_pct,estimated_click_uplift,inlinks,link_score,word_count
0,14,catalog,37429,18.0,1.23,288.0,38526.0,98.0,489.0
1,87,catalog,7271,17.0,1.36,47.0,78.0,1.0,1825.0
2,66,product,6097,17.0,1.95,3.0,129.0,1.0,1046.0
3,183,catalog,6084,12.0,0.92,66.0,52.0,1.0,3893.0
4,473,product,4134,20.0,0.53,61.0,217.0,2.0,917.0
5,152,catalog,3806,12.0,1.68,12.0,36.0,1.0,3886.0
6,1954,catalog,2733,18.0,0.11,52.0,37.0,1.0,1118.0
7,333,catalog,2314,13.0,1.38,14.0,80.0,1.0,4738.0
8,262,catalog,2288,16.0,1.75,6.0,80.0,1.0,4427.0
9,430,product,2282,17.0,1.10,21.0,67.0,1.0,1089.0


#### 1.3.3 Internal Linking & Link Equity

In [382]:
# Question 7: Is link equity well spread?

query = """
SELECT
    CASE
        WHEN link_score = 0             THEN '0 — no equity'
        WHEN link_score BETWEEN 1 AND 5 THEN '1–5'
        WHEN link_score BETWEEN 6 AND 20 THEN '6–20'
        WHEN link_score BETWEEN 21 AND 50 THEN '21–50'
        ELSE                                 '51–99 — high equity'
    END AS link_score_band,
    COUNT(*)                            AS pages,
    ROUND(AVG(clicks), 1)               AS avg_clicks,
    ROUND(AVG(position), 2)             AS avg_position,
    ROUND(AVG(ctr) * 100, 2)           AS avg_ctr_pct
FROM vw_page_base
WHERE link_score IS NOT NULL
GROUP BY link_score_band
ORDER BY MIN(link_score);
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_7.html


,link_score_band,pages,avg_clicks,avg_position,avg_ctr_pct
0,0 — no equity,11,0.6,27.09,5.15
1,1–5,9103,5.2,14.09,1.94
2,6–20,28,8.7,15.25,1.94
3,21–50,7,1.6,15.14,5.73
4,51–99 — high equity,53,66.6,21.81,0.81


In [383]:
# Question 8: Which pages are poorly linked with good ranking potential?

query = """
SELECT
    page_id,
    segment,
    clicks,
    impressions,
    ROUND(position, 1)                  AS position,
    inlinks,
    link_score,
    internal_link_flag
FROM vw_content_health
WHERE internal_link_flag IN ('orphaned', 'poorly_linked')
  AND position <= 20
  AND impressions >= 50
ORDER BY impressions DESC
LIMIT 20;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_8.html


""


#### 1.3.4 Crawl Budget Analysis

In [385]:
# Question 9: How is the site's overall crawl budget being allocated?

query = """
SELECT
    crawl_efficiency_flag,
    COUNT(*)                                AS pages,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_pages,
    SUM(robots_visits)                      AS total_robot_visits,
    ROUND(100.0 * SUM(robots_visits)
          / NULLIF(SUM(SUM(robots_visits)) OVER (), 0), 1) AS pct_robot_visits,
    SUM(clicks)                             AS total_clicks
FROM vw_crawl_efficiency
GROUP BY crawl_efficiency_flag
ORDER BY total_robot_visits DESC;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_9.html


,crawl_efficiency_flag,pages,pct_pages,total_robot_visits,pct_robot_visits,total_clicks
0,efficient,8392,88.9,5712,71.5,37844
1,inefficient,9,0.1,1286,16.1,50
2,high_waste,9,0.1,843,10.5,0
3,low_waste,64,0.7,150,1.9,0
4,not_crawled,965,10.2,0,0.0,18550


In [386]:
# Question 10: What are the top 20 pages wasting crawl budget and are candidates for noindex, consolidation, or robots.txt exclusion?

query = """
SELECT
    page_id,
    segment,
    robots_visits,
    clicks,
    impressions,
    ROUND(position, 1)                  AS position,
    word_count,
    inlinks,
    response_time_flag
FROM vw_crawl_efficiency
WHERE crawl_efficiency_flag = 'high_waste'
ORDER BY robots_visits DESC
LIMIT 20;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_10.html


,page_id,segment,robots_visits,clicks,impressions,position,word_count,inlinks,response_time_flag
0,5629,catalog,469,0,37,16.0,1539,45,slow
1,6504,catalog,115,0,8,56.0,3572,34,slow
2,8107,catalog,90,0,1,8.0,4547,111,slow
3,8193,catalog,54,0,17,29.0,4508,91,slow
4,8180,catalog,37,0,333,13.0,3663,36,slow
5,7210,catalog,28,0,1,3.0,3608,32,slow
6,6465,catalog,24,0,61,21.0,5309,29142,slow
7,4704,catalog,15,0,11,15.0,748,29,slow
8,4849,catalog,11,0,480,21.0,1933,94,slow


#### 1.3.5 Content & Technical Health

In [388]:
# Question 11: What are the most common on-page problems and how frequent are these content issues?

query = """
SELECT 'Title too short'         AS issue, SUM(CASE WHEN title_length_flag  = 'too_short' THEN 1 ELSE 0 END) AS pages FROM vw_content_health
UNION ALL
SELECT 'Title too long',                   SUM(CASE WHEN title_length_flag  = 'too_long'  THEN 1 ELSE 0 END) FROM vw_content_health
UNION ALL
SELECT 'Meta description too short',       SUM(CASE WHEN meta_desc_flag     = 'too_short' THEN 1 ELSE 0 END) FROM vw_content_health
UNION ALL
SELECT 'Meta description too long',        SUM(CASE WHEN meta_desc_flag     = 'too_long'  THEN 1 ELSE 0 END) FROM vw_content_health
UNION ALL
SELECT 'H1 absent',                        SUM(CASE WHEN h1_flag            = 'absent'    THEN 1 ELSE 0 END) FROM vw_content_health
UNION ALL
SELECT 'Thin content (<200 words)',         SUM(CASE WHEN content_depth_flag = 'thin'      THEN 1 ELSE 0 END) FROM vw_content_health
UNION ALL
SELECT 'Slow response time (>2s)',          SUM(CASE WHEN response_time_flag = 'slow'      THEN 1 ELSE 0 END) FROM vw_content_health
UNION ALL
SELECT 'Orphaned (0 inlinks)',              SUM(CASE WHEN internal_link_flag = 'orphaned'  THEN 1 ELSE 0 END) FROM vw_content_health
ORDER BY pages DESC;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_11.html


,issue,pages
0,Slow response time (>2s),9196
1,Title too long,9106
2,Meta description too long,8548
3,H1 absent,76
4,Meta description too short,70
5,Thin content (<200 words),59
6,Title too short,34
7,Orphaned (0 inlinks),2


In [389]:
# Question 12: Which pages have 3 or more issues at once, sorted by organic visibility?

query = """
SELECT
    page_id,
    segment,
    impressions,
    ROUND(position, 1)          AS position,
    issue_count,
    title_length_flag,
    meta_desc_flag,
    h1_flag,
    content_depth_flag,
    internal_link_flag,
    response_time_flag
FROM vw_content_health
WHERE issue_count >= 3
ORDER BY issue_count DESC, impressions DESC
LIMIT 25;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_12.html


,page_id,segment,impressions,position,issue_count,title_length_flag,meta_desc_flag,h1_flag,content_depth_flag,internal_link_flag,response_time_flag
0,2870,brands,46,26.0,6,too_short,too_short,absent,thin,orphaned,slow
1,2883,brands,17,11.0,6,too_short,too_short,absent,thin,orphaned,slow
2,233,uncategorised,3164,23.0,5,too_short,too_short,absent,thin,poorly_linked,slow
3,34,uncategorised,1256,12.0,5,too_long,too_long,absent,thin,well_linked,slow
4,3489,catalog,183,27.0,5,too_short,too_short,absent,thin,well_linked,slow
5,2263,uncategorised,162,8.0,5,too_short,too_short,absent,thin,poorly_linked,slow
6,6558,catalog,126,33.0,5,too_short,too_short,absent,thin,poorly_linked,slow
7,4169,uncategorised,105,50.0,5,too_short,too_short,absent,thin,poorly_linked,slow
8,2905,uncategorised,103,8.0,5,too_short,too_short,absent,thin,poorly_linked,slow
9,8958,uncategorised,58,35.0,5,too_short,too_short,absent,thin,poorly_linked,slow


In [390]:
# Question 13: Do slow pages rank worse?

query = """
SELECT
    response_time_flag,
    segment,
    COUNT(*)                        AS pages,
    ROUND(AVG(position), 2)         AS avg_position,
    ROUND(AVG(ctr) * 100, 2)       AS avg_ctr_pct,
    ROUND(AVG(bounce_rate) * 100, 2) AS avg_bounce_pct,
    ROUND(AVG(clicks), 1)           AS avg_clicks
FROM vw_page_base
WHERE response_time_flag != 'unknown'
GROUP BY response_time_flag, segment
ORDER BY avg_position;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_13.html


,response_time_flag,segment,pages,avg_position,avg_ctr_pct,avg_bounce_pct,avg_clicks
0,fast,catalog,1,3.00,0.00,NaN,0.0
1,acceptable,product,1,4.00,0.00,NaN,0.0
2,slow,product,3503,10.67,2.13,0.0,7.5
3,slow,brands,710,13.61,2.16,0.0,2.4
4,slow,uncategorised,177,15.24,5.45,NaN,24.6
5,slow,catalog,4806,16.74,1.64,0.0,4.0
6,acceptable,catalog,4,24.25,1.33,NaN,3.0


#### 1.3.6 Engagement Exploration

In [392]:
# Question 14: Do higher-ranking pages engage users better?

query = """
SELECT
    position_bucket,
    COUNT(*)                                    AS pages,
    ROUND(AVG(bounce_rate) * 100, 1)           AS avg_bounce_pct,
    ROUND(AVG(view_depth), 2)                   AS avg_view_depth,
    ROUND(AVG(time_spent_seconds), 0)           AS avg_time_on_page_s,
    ROUND(AVG(mobility) * 100, 1)              AS avg_mobile_pct
FROM vw_page_base
WHERE bounce_rate IS NOT NULL   -- only rows with analytics data
GROUP BY position_bucket
ORDER BY position_bucket;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_14.html


,position_bucket,pages,avg_bounce_pct,avg_view_depth,avg_time_on_page_s,avg_mobile_pct
0,2. Page 1 (4-10),12,8.3,1.0,52.0,100.0


In [393]:
# Question 15: What is the response time of high-traffic pages?

query = """
SELECT
    page_id,
    segment,
    clicks,
    response_time_flag,
    word_count,
    ROUND(position, 1) AS avg_position
FROM vw_page_base
WHERE clicks > 200
ORDER BY clicks DESC;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_15.html


,page_id,segment,clicks,response_time_flag,word_count,avg_position
0,1,uncategorised,1939,slow,1553,8.0
1,2,product,1798,slow,950,8.0
2,3,product,951,slow,858,6.0
3,4,product,863,slow,910,7.0
4,5,uncategorised,822,slow,528,17.0
5,6,catalog,684,slow,3878,21.0
6,7,product,595,slow,1157,3.0
7,8,catalog,550,slow,4671,14.0
8,9,product,542,slow,836,7.0
9,10,product,533,slow,1272,10.0


#### 1.3.7 Data Export Queries

In [395]:
# 16: Data export of analysis-ready dataset for Python

query = """
SELECT
    page_id,
    segment,
    clicks,
    impressions,
    position,
    ROUND(ctr, 4)                   AS ctr,
    bounce_rate,
    view_depth,
    time_spent_seconds,
    robots_visits,
    mobility,
    title_length,
    meta_description_length,
    h1_length,
    word_count,
    sentence_count,
    folder_depth,
    link_score,
    inlinks,
    outlinks,
    response_time,
    position_bucket,
    title_length_flag,
    meta_desc_flag,
    h1_flag,
    response_time_flag
FROM vw_page_base
ORDER BY clicks DESC;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_16.html


,page_id,segment,clicks,impressions,position,ctr,bounce_rate,view_depth,time_spent_seconds,robots_visits,...,folder_depth,link_score,inlinks,outlinks,response_time,position_bucket,title_length_flag,meta_desc_flag,h1_flag,response_time_flag
0,1,uncategorised,1939,41950,"8,6",0.0462,NaN,NaN,690.0,837.0,...,0.0,99.0,24095.0,334.0,"2,185",5. Deep (51+),too_long,too_long,absent,slow
1,2,product,1798,27042,"8,2",0.0665,NaN,NaN,41.0,7.0,...,3.0,2.0,173.0,77.0,"1,332",5. Deep (51+),too_long,too_long,present,slow
2,3,product,951,18076,"6,3",0.0526,NaN,NaN,53.0,3.0,...,3.0,1.0,143.0,77.0,"1,759",5. Deep (51+),too_long,too_long,present,slow
3,4,product,863,10697,"7,9",0.0807,NaN,NaN,43.0,3.0,...,3.0,1.0,143.0,77.0,"1,495",5. Deep (51+),too_long,optimal,present,slow
4,5,uncategorised,822,7269,"17,8",0.1131,NaN,NaN,139.0,27.0,...,2.0,2.0,120.0,334.0,"0,18",5. Deep (51+),too_long,too_long,present,slow
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9434,4372,catalog,0,1,5.0,0.0000,NaN,NaN,NaN,NaN,...,3.0,1.0,41.0,191.0,"1,722",2. Page 1 (4-10),too_long,too_long,present,slow
9435,4366,catalog,0,1,13.0,0.0000,NaN,NaN,NaN,NaN,...,4.0,1.0,32.0,192.0,"1,57",3. Page 2 (11-20),too_long,too_long,present,slow
9436,4361,catalog,0,1,20.0,0.0000,NaN,NaN,NaN,NaN,...,4.0,1.0,27.0,186.0,"1,425",3. Page 2 (11-20),too_long,too_long,present,slow
9437,4347,catalog,0,1,36.0,0.0000,NaN,NaN,NaN,NaN,...,4.0,1.0,32.0,192.0,"1,838",4. Mid (21-50),too_long,too_long,present,slow


In [396]:
# 17: Data export for R

query = """
SELECT
    clicks,
    impressions,
    position,
    ROUND(ctr, 4)                   AS ctr,
    bounce_rate,
    view_depth,
    robots_visits,
    mobility,
    title_length,
    meta_description_length,
    h1_length,
    word_count,
    sentence_count,
    folder_depth,
    link_score,
    inlinks,
    outlinks,
    response_time,
    segment
FROM vw_page_base
WHERE position        IS NOT NULL
  AND title_length    IS NOT NULL
  AND word_count      IS NOT NULL
  AND inlinks         IS NOT NULL
  AND response_time   IS NOT NULL
ORDER BY clicks DESC;
"""

result = %sql $query

save_table(result)

result

 * sqlite:///seo_data.db
Done.
Saved: sql_report_tables/table_17.html


,clicks,impressions,position,ctr,bounce_rate,view_depth,robots_visits,mobility,title_length,meta_description_length,h1_length,word_count,sentence_count,folder_depth,link_score,inlinks,outlinks,response_time,segment
0,1939,41950,"8,6",0.0462,NaN,NaN,837.0,NaN,83,188,0,1553,367,0,99.0,24095,334,"2,185",uncategorised
1,1798,27042,"8,2",0.0665,NaN,NaN,7.0,NaN,152,227,37,950,377,3,2.0,173,77,"1,332",product
2,951,18076,"6,3",0.0526,NaN,NaN,3.0,NaN,101,159,38,858,341,3,1.0,143,77,"1,759",product
3,863,10697,"7,9",0.0807,NaN,NaN,3.0,NaN,70,137,24,910,359,3,1.0,143,77,"1,495",product
4,822,7269,"17,8",0.1131,NaN,NaN,27.0,NaN,85,181,24,528,103,2,2.0,120,334,"0,18",uncategorised
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9197,0,1,5.0,0.0000,NaN,NaN,NaN,NaN,83,184,18,839,375,3,1.0,41,191,"1,722",catalog
9198,0,1,13.0,0.0000,NaN,NaN,NaN,NaN,127,206,40,845,375,4,1.0,32,192,"1,57",catalog
9199,0,1,20.0,0.0000,NaN,NaN,NaN,NaN,100,194,25,797,355,4,1.0,27,186,"1,425",catalog
9200,0,1,36.0,0.0000,NaN,NaN,NaN,NaN,106,195,30,842,376,4,1.0,32,192,"1,838",catalog
